In [ ]:
import ctypes
import sys
import time
import threading

import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# 設定 pyautogui 參數
pyautogui.PAUSE = 0.01
pyautogui.FAILSAFE = False

# =========================
# 自動連點器設定 (每秒 30 次 + 拖曳避讓)
# =========================
AUTO_CLICK_RUNNING = True   # 控制連點總開關
ALLOW_AUTO_CLICK = True     # 控制拖曳時暫停連點

def auto_clicker():
    """在背景獨立運作的連點器，拖曳寶石時會自動暫停"""
    interval = 1.0 / 30.0  # 約 0.0333 秒
    while AUTO_CLICK_RUNNING:
        if ALLOW_AUTO_CLICK:
            pyautogui.click(button='left')
        time.sleep(interval)

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 498, 408
RIGHT, BOTTOM = 817, 730

WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# 寶石中心點座標計算
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# 預先算好 8x8 各格中心周圍 3x3 區域的相對 (x, y) 座標
SAMPLE_OFFSETS = []
for r in range(N):
    row_offsets = []
    cy = int((r + 0.5) * CH)
    for c in range(N):
        cx = int((c + 0.5) * CW)
        pts = [
            (cx + dx, cy + dy)
            for dy in (-3, 0, 3)
            for dx in (-3, 0, 3)
        ]
        row_offsets.append(pts)
    SAMPLE_OFFSETS.append(row_offsets)

# =========================
# 流程與反跳針設定
# =========================
ROUND_TIME = 80          # 延長至 80 秒
START_DELAY = 2          
MAX_ATTEMPTS_PER_MOVE = 2 # 同組合無效 2 次即換步

IGNORE = {"empty", "unknown"}
move_fail_count = {}
recent_moves_history = [] # 紀錄近期執行的動作歷史


# =========================
# 高正確率色彩辨識（RGB 3x3 多點平均）
# =========================
def identify_rgb(r, g, b):
    """利用多點平均後的 RGB 特性判定寶石顏色"""
    if r < 50 and g < 50 and b < 50:
        return "empty"
    if r > 175 and g > 175 and b > 175 and abs(r - g) < 25 and abs(g - b) < 25:
        return "white"
    
    if r > 140 and g < 80 and b < 80:
        return "red"
    if r > 170 and g > 95 and b < 65:
        return "orange"
    if r > 160 and g > 160 and b < 85:
        return "yellow"
    if g > r and g > b:
        return "green"
    if b > r and b > g:
        return "blue"
    if r > 95 and b > 95 and g < 95:
        return "purple"

    return "unknown"


def fast_read_board():
    """多點取樣盤面讀取（耗時約 20ms）"""
    img = ImageGrab.grab(bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True)
    getpixel = img.getpixel
    
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            pts = SAMPLE_OFFSETS[r][c]
            sr = sg = sb = 0
            for px in pts:
                rgb = getpixel(px)
                sr += rgb[0]
                sg += rgb[1]
                sb += rgb[2]
            
            row.append(identify_rgb(sr // 9, sg // 9, sb // 9))
        board.append(row)
    return board


def stop_pressed():
    """按 Q 鍵強制停止"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 消除與尋步邏輯
# =========================
def find_matches(board):
    found = set()
    # 水平消除
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直消除
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


def best_move(board):
    candidates = []

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE or color == other:
                    continue

                move_key = tuple(sorted([(r, c), (r2, c2)]))
                
                # 排除失敗計數達到上限的組合
                if move_fail_count.get(move_key, 0) >= MAX_ATTEMPTS_PER_MOVE:
                    continue

                # 模擬交換
                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color

                found = find_matches(test)
                if (r, c) not in found and (r2, c2) not in found:
                    continue

                score = len(found)
                candidates.append((score, ((r, c), (r2, c2)), move_key))

    if not candidates:
        return None

    # 按消除數量排序
    candidates.sort(key=lambda item: item[0], reverse=True)
    best_score, best_move_pair, move_key = candidates[0]
    return best_move_pair, best_score, move_key


def board_is_empty(board):
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


def swap(a, b):
    global ALLOW_AUTO_CLICK
    # 執行拖曳時暫停連點，避免點擊干擾滑鼠拖曳
    ALLOW_AUTO_CLICK = False
    
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.10, button="left")
    
    # 拖曳完成後恢復連點
    ALLOW_AUTO_CLICK = True


# =========================
# 主流程
# =========================
def main():
    global AUTO_CLICK_RUNNING

    print("寶石方塊 防死循環鎖定版 (80秒全自動運作)")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"預計執行 80 秒（按 Q 鍵可手動中斷）")
    print(f"{START_DELAY} 秒後開始...")
    time.sleep(START_DELAY)

    # 啟動背景連點 Thread
    clicker_thread = threading.Thread(target=auto_clicker, daemon=True)
    clicker_thread.start()
    print("-> 自動連點器已開始運作（每秒 30 次）")

    start = time.time()
    moves = 0

    try:
        while True:
            if stop_pressed():
                print("\n[手動中斷] 已按下 Q 鍵停止")
                break

            elapsed = time.time() - start
            if elapsed >= ROUND_TIME:
                print(f"\n[時間到達] 已執行滿 {elapsed:.1f} 秒，自動停止")
                break

            # 1. 抓取盤面
            t0 = time.time()
            board = fast_read_board()

            if board_is_empty(board):
                time.sleep(0.1)
                continue

            # 2. 搜尋有效步
            result = best_move(board)
            calc_time = (time.time() - t0) * 1000

            # 當前沒有步數可用時
            if result is None:
                if len(move_fail_count) > 0:
                    print(f"[{elapsed:5.1f}s] 無可用組合，清空黑名單並重新掃描...")
                    move_fail_count.clear()
                    recent_moves_history.clear()
                time.sleep(0.15)
                continue

            (a, b), score, move_key = result

            # 3. 強化版反死循環檢測 (Anti-loop Hard Lock)
            recent_moves_history.append(move_key)
            if len(recent_moves_history) > 8:
                recent_moves_history.pop(0)

            # 只要在最近 8 次操作中，同一個動作出現超過 3 次，就強制鎖定
            if recent_moves_history.count(move_key) >= 3:
                print(f"  └─> ⚠️ 偵測到重複無效動作 {move_key}，強制鎖定此組合！")
                move_fail_count[move_key] = MAX_ATTEMPTS_PER_MOVE
                continue

            print(f"[{elapsed:5.1f}s] (判斷: {calc_time:.1f}ms) 交換 {a} <-> {b} (預估消 {score} 顆, 失敗累積: {move_fail_count.get(move_key, 0)})")
            
            # 4. 執行動作與紀錄驗證
            swap(a, b)
            moves += 1
            
            # 等待動畫穩定
            time.sleep(0.25)
            new_board = fast_read_board()
            
            # 比較操作前後的盤面
            if signature(new_board) == signature(board):
                move_fail_count[move_key] = move_fail_count.get(move_key, 0) + 1
                print(f"  └> 組合 {move_key} 未能消除，失敗計數 +1 (目前: {move_fail_count[move_key]})")
            else:
                # 盤面真正發生改變時，才清除當前組合的失敗記錄
                move_fail_count.pop(move_key, None)

    finally:
        AUTO_CLICK_RUNNING = False
        print("-> 自動連點器已關閉")

    print(f"\n遊戲結束：共執行交換 {moves} 次，總耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    main()

寶石方塊 防死循環鎖定版 (含連點衝突避讓)
棋盤範圍：x 498~817, y 408~730
跑滿 70 秒（按 Q 鍵可停止）
2 秒後開始...
-> 自動連點器已開始運作（每秒 30 次）
[  0.0s] (判斷: 45.9ms) 交換 (6, 4) <-> (6, 5) (預估消 7 顆, 失敗累積: 0)
[  0.3s] (判斷: 28.1ms) 交換 (3, 3) <-> (3, 4) (預估消 14 顆, 失敗累積: 0)
[  0.7s] (判斷: 31.5ms) 交換 (4, 2) <-> (5, 2) (預估消 16 顆, 失敗累積: 0)
[  1.0s] (判斷: 28.0ms) 交換 (6, 4) <-> (6, 5) (預估消 15 顆, 失敗累積: 0)
[  1.3s] (判斷: 28.8ms) 交換 (4, 2) <-> (5, 2) (預估消 15 顆, 失敗累積: 0)
[  1.7s] (判斷: 30.0ms) 交換 (1, 3) <-> (2, 3) (預估消 14 顆, 失敗累積: 0)
[  2.0s] (判斷: 26.0ms) 交換 (0, 3) <-> (1, 3) (預估消 17 顆, 失敗累積: 0)
  └─> ⚠️ 偵測到重複無效動作 ((4, 2), (5, 2))，強制鎖定此組合！
[  2.4s] (判斷: 27.6ms) 交換 (5, 3) <-> (5, 4) (預估消 16 顆, 失敗累積: 0)
[  2.7s] (判斷: 28.8ms) 交換 (5, 2) <-> (5, 3) (預估消 14 顆, 失敗累積: 0)
[  3.0s] (判斷: 26.4ms) 交換 (4, 1) <-> (5, 1) (預估消 13 顆, 失敗累積: 0)
[  3.4s] (判斷: 29.0ms) 交換 (2, 1) <-> (2, 2) (預估消 15 顆, 失敗累積: 0)
[  3.7s] (判斷: 28.8ms) 交換 (4, 1) <-> (5, 1) (預估消 15 顆, 失敗累積: 0)
[  4.0s] (判斷: 27.5ms) 交換 (2, 1) <-> (2, 2) (預估消 15 顆, 失敗累積: 0)
  └─> ⚠️ 偵測到重複無效動作 ((4, 1), (5, 1))，強

在@game bot/bilezbot.ipynb 中新增一個 code cell,完成底下的程式。
basex以及basey 是左上角寶石中心點的位置。畫面中共有8x8顆寶石。 每顆寶石的間距指正在gap 中。
寫一個函式,接受一個row(用r代表),一個column(用c代表),回傳(r, c)所代表的寶石的位置

import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 495, 371
RIGHT, BOTTOM = 820, 691
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 62          # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 10         # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8          # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 2.0     # 等棋盤穩定的最長時間（秒）
VERBOSE = True           # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
def identify(patch):
    """把一小塊畫面（BGR）轉成寶石顏色名稱。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h, s, v = (int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))

    if v < 55:                      # 太暗 → 空格
        return "empty"
    if s < 45:                      # 幾乎沒有飽和度 → 白色 / 特殊寶石
        return "white"
    if h < 8 or h >= 170:
        return "red"
    if h < 22:
        return "orange"
    if h < 35:
        return "yellow"
    if h < 85:
        return "green"
    if h < 115:
        return "blue"
    return "purple"


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - 4:y + 5, x - 4:x + 5]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print("  " + " ".join(SYMBOL.get(c, "?") for c in row))


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def best_move(board):
    """回傳 (a, b, score)；找不到就回傳 None。"""
    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE:
                    continue
                if color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color

                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if (r, c) not in found and (r2, c2) not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = ((r, c), (r2, c2), score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV（h, s, v）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            h, s, v = (int(n) for n in np.median(
                hsv[y - 4:y + 5, x - 4:x + 5].reshape(-1, 3), axis=0))
            print(f"  ({r},{c}) -> {board[r][c]:7s} h={h:3d} s={s:3d} v={v:3d}")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", img)
    if ok:
        buf.tofile(path)      # 用 tofile 才支援中文路徑
        print(f"\n截圖已存成 {path}")


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board)

        if move is None:
            still += 1
            print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")
            if still >= STILL_LIMIT:
                print("連續找不到步可走，判定遊戲結束")
                break
            time.sleep(0.4)
            continue

        still = 0
        a, b, score = move
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if _name_ == "_main_":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "selftest":
        selftest()
    else:
        main()


座標改X1=308 y1=375
x2= 629 y2=693